# Parte II — Preparación de datos

**CC3084 — Data Science — Laboratorio 4**
**Universidad del Valle de Guatemala — Semestre II, 2026**

Este notebook verifica los rásteres NDVI y NDWI generados en la **Parte I** (`src/descargar_indices.py`) antes de utilizarlos en el análisis de la Parte II. No vuelve a descargar nada de Copernicus: solo lee lo que ya existe en `data/indices/`.

## Objetivo

El objetivo de este notebook es el de explorar la informacion de los rasters obtenidos en la parte 1 del laboratorio para verificar la integridad de los datos para podteriormente hacer preparacion y tratamiento de los datos para convertirtlos en un data frame que pueda alimentar modelos de DL como de ML. El dataframe consistira en que los pixeles de las imagenes seran als filas con diferente informacion relevante como su posicion o indice de cianobacterias.

## Preparación del entorno

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import rasterio.warp
from matplotlib.colors import ListedColormap

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "figure.dpi": 110,
})

DATA_DIR = Path("../../data/indices")
CYANO_DIR = Path("../../notebooks/resultados_cyano")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Área y fechas oficiales

Mismas cajas delimitadoras (EPSG:4326) y fechas oficiales usadas en la Parte I, para verificar que ningún raster falte.

In [ ]:
FECHAS_ATITLAN = [
    "2025-01-18", "2025-04-13", "2025-05-13", "2025-07-17",
    "2025-11-21", "2025-12-29", "2026-02-12", "2026-03-24",
    "2026-04-13", "2026-04-28", "2026-07-22",
]
FECHAS_AMATITLAN = [
    "2025-01-28", "2025-04-15", "2025-04-28", "2025-11-24",
    "2026-01-08", "2026-02-02", "2026-02-07", "2026-03-29",
    "2026-04-13", "2026-04-28", "2026-06-19",
]
FECHAS_ESPERADAS = {"atitlan": FECHAS_ATITLAN, "amatitlan": FECHAS_AMATITLAN}

print(f"Fechas esperadas — Atitlán: {len(FECHAS_ATITLAN)}, Amatitlán: {len(FECHAS_AMATITLAN)}")

## Inventario de archivos

Se listan los GeoTIFF de `data/indices/` y se extrae lago, fecha e índice a partir del nombre de archivo (`<lago>_<fecha>_<indice>.tif`).

In [ ]:
PATRON = re.compile(r"(?P<lago>atitlan|amatitlan)_(?P<fecha>\d{4}-\d{2}-\d{2})_(?P<indice>ndvi|ndwi)\.tif$")

archivos = sorted(DATA_DIR.glob("*.tif"))

registros = []
for ruta in archivos:
    coincidencia = PATRON.match(ruta.name)
    if not coincidencia:
        raise ValueError(f"Nombre de archivo inesperado: {ruta.name}")
    registros.append({**coincidencia.groupdict(), "archivo": ruta.name, "ruta": ruta})

inventario = pd.DataFrame(registros)
inventario["fecha"] = pd.to_datetime(inventario["fecha"])

print(f"Rásteres encontrados en {DATA_DIR}: {len(inventario)}")
inventario.head()

### Completitud

Se espera exactamente un NDVI y un NDWI por cada fecha oficial de cada lago (44 combinaciones en total).

In [ ]:
conteo_por_lago_indice = inventario.groupby(["lago", "indice"]).size().unstack("indice")
conteo_por_lago_indice

In [ ]:
faltantes = []
for lago, fechas in FECHAS_ESPERADAS.items():
    for fecha in fechas:
        for indice in ("ndvi", "ndwi"):
            existe = (
                (inventario["lago"] == lago)
                & (inventario["fecha"] == pd.Timestamp(fecha))
                & (inventario["indice"] == indice)
            ).any()
            if not existe:
                faltantes.append((lago, fecha, indice))

assert not faltantes, f"Faltan combinaciones lago/fecha/índice: {faltantes}"
print(f"Combinaciones esperadas: {sum(len(f) for f in FECHAS_ESPERADAS.values()) * 2}")
print(f"Combinaciones encontradas: {len(inventario)}")
print("Inventario completo: no faltan rásteres." if not faltantes else "Faltan rásteres.")

## Metadatos ráster

Para cada archivo se lee su CRS, dimensiones, resolución, tipo de dato y cobertura de píxeles válidos (no `NaN`), además del rango de valores observado.

In [ ]:
def leer_metadatos(ruta: Path) -> dict:
    """Lee metadatos y estadísticas básicas de un GeoTIFF de un solo índice."""
    with rasterio.open(ruta) as src:
        datos = src.read(1)
        valido = ~np.isnan(datos)
        valores_validos = datos[valido]
        fuera_de_rango = np.sum((valores_validos < -1) | (valores_validos > 1)) if valido.any() else 0
        return {
            "crs": str(src.crs),
            "ancho": src.width,
            "alto": src.height,
            "resolucion_x": src.res[0],
            "resolucion_y": src.res[1],
            "dtype": src.dtypes[0],
            "pixeles_totales": int(datos.size),
            "pixeles_validos": int(valido.sum()),
            "minimo": float(valores_validos.min()) if valido.any() else np.nan,
            "maximo": float(valores_validos.max()) if valido.any() else np.nan,
            "promedio": float(valores_validos.mean()) if valido.any() else np.nan,
            "fuera_de_rango": int(fuera_de_rango),
        }

metadatos = pd.DataFrame(inventario["ruta"].apply(leer_metadatos).tolist())
inventario = pd.concat([inventario.reset_index(drop=True), metadatos], axis=1)
inventario["pct_validos"] = 100 * inventario["pixeles_validos"] / inventario["pixeles_totales"]
inventario["pct_fuera_de_rango"] = 100 * inventario["fuera_de_rango"] / inventario["pixeles_validos"]

inventario.head()

### Consistencia de CRS, resolución y tipo de dato

Todos los rásteres deben compartir sistema de referencia, resolución espacial y tipo de dato; solo el ancho/alto puede variar entre lagos porque sus cajas delimitadoras son distintas.

In [ ]:
assert inventario["crs"].nunique() == 1, "Los rásteres no comparten el mismo CRS"
assert inventario["dtype"].nunique() == 1, "Los rásteres no comparten el mismo tipo de dato"
assert set(inventario[["resolucion_x", "resolucion_y"]].itertuples(index=False)) == {(10.0, 10.0)}, \
    "Se esperaba una resolución uniforme de 10 x 10 m"

for lago, grupo in inventario.groupby("lago"):
    dimensiones = grupo[["ancho", "alto"]].drop_duplicates()
    assert len(dimensiones) == 1, f"Dimensiones inconsistentes dentro de {lago}"
    print(f"{lago}: {dimensiones.iloc[0]['ancho']} x {dimensiones.iloc[0]['alto']} px, CRS {grupo['crs'].iloc[0]}")

print(f"\nResolución uniforme: 10 x 10 m — tipo de dato uniforme: {inventario['dtype'].iloc[0]}")

## Cobertura de píxeles válidos

Los archivos se descargan sobre la caja delimitadora completa de cada lago, por lo que un porcentaje de `NaN` corresponde a tierra/nubes fuera de la máscara de openEO. Se listan los diez rásteres con menor cobertura válida.

In [ ]:
inventario.sort_values("pct_validos")[["archivo", "pixeles_validos", "pixeles_totales", "pct_validos"]].head(10)

## Valores fuera del rango físico de NDVI/NDWI

`NDVI` y `NDWI` son diferencias normalizadas y deben quedar acotadas en `[-1, 1]`. A diferencia de `src/procesar_indices.py` (que sí enmascara denominadores cercanos a cero), `src/descargar_indices.py` calcula los índices directamente en openEO sin ese filtro, así que algunos píxeles —típicamente bordes de nube o de la caja delimitadora, donde `B08 + B04` o `B03 + B08` se acercan a cero— pueden salir del rango esperado. Antes de usarlos en la Parte II conviene enmascararlos.

In [ ]:
resumen_rango = (
    inventario[inventario["fuera_de_rango"] > 0]
    .sort_values("pct_fuera_de_rango", ascending=False)
    [["archivo", "lago", "fecha", "indice", "fuera_de_rango", "pct_fuera_de_rango", "minimo", "maximo"]]
)

print(f"Archivos con píxeles fuera de [-1, 1]: {len(resumen_rango)} de {len(inventario)}")
resumen_rango.head(10)

### Caso extremo: antes y después de enmascarar

Se visualiza el raster con mayor proporción de píxeles fuera de rango, comparado contra la misma capa enmascarando a `NaN` todo valor fuera de `[-1, 1]`.

In [ ]:
peor = resumen_rango.iloc[0]

with rasterio.open(inventario.loc[inventario["archivo"] == peor["archivo"], "ruta"].iloc[0]) as src:
    original = src.read(1)

enmascarado = np.where((original < -1) | (original > 1), np.nan, original)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, capa, titulo in zip(axes, [original, enmascarado], ["Original (Parte I)", "Enmascarado a [-1, 1]"]):
    im = ax.imshow(capa, cmap="RdYlGn", vmin=-1, vmax=1)
    ax.set_title(titulo)
    ax.axis("off")

fig.suptitle(f"{peor['archivo']} — {peor['fuera_de_rango']} píxeles fuera de rango ({peor['pct_fuera_de_rango']:.3f} %)")
fig.colorbar(im, ax=axes, shrink=0.7, label=peor["indice"].upper())
plt.show()

## Muestra visual por lago

Vista rápida de NDVI y NDWI para ambos lagos en una fecha compartida (2026-04-28), la que menos píxeles fuera de rango presenta en el inventario.

In [ ]:
fecha_muestra = "2026-04-28"
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for fila, lago in enumerate(["atitlan", "amatitlan"]):
    for columna, indice in enumerate(["ndvi", "ndwi"]):
        ruta = inventario.loc[
            (inventario["lago"] == lago)
            & (inventario["fecha"] == pd.Timestamp(fecha_muestra))
            & (inventario["indice"] == indice),
            "ruta",
        ].iloc[0]
        with rasterio.open(ruta) as src:
            datos = src.read(1)

        ax = axes[fila, columna]
        cmap = "RdYlGn" if indice == "ndvi" else "Blues"
        im = ax.imshow(datos, cmap=cmap, vmin=-1, vmax=1)
        ax.set_title(f"{lago.capitalize()} — {indice.upper()} ({fecha_muestra})")
        ax.axis("off")
        fig.colorbar(im, ax=ax, shrink=0.75)

plt.tight_layout()
plt.show()

## Exportar inventario para la Parte II

Se guarda el inventario con metadatos y estadísticas por raster para que los siguientes notebooks de la Parte II no tengan que releer todos los GeoTIFF.

In [ ]:
inventario_export = inventario.drop(columns=["ruta"]).copy()
inventario_export["fecha"] = inventario_export["fecha"].dt.strftime("%Y-%m-%d")

ruta_salida = OUTPUT_DIR / "inventario_rasters.csv"
inventario_export.sort_values(["lago", "fecha", "indice"]).to_csv(ruta_salida, index=False)

print(f"Inventario guardado en {ruta_salida.resolve()}")
inventario_export.head()

### Ahora que ya comprobamos que toda la informacion de los rasters esta integra, ya verificamos los datos que nos interesaban y los que no como la vegetacion y nubosidad. Ahora vamos a proceder a convertir esta informacion a un conjunto de datos validos para modelos de ML y DL para responder a una variable respeusta que ams adelante vamos a formalizar

###  Enriquecimiento: bandas, coordenadas geográficas y cianobacteria

Antes de limpiar los píxeles se agregan al dataset columnas para enriquecer el aprendizaje de los modelos reutilizando información que ya existe en la primera parte del laboratorio:

- **Coordenadas**: además de `x`/`y` (UTM, `EPSG:32615`, la proyección nativa de los rásteres) se agregan `lon`/`lat` en `EPSG:4326`, más fáciles de interpretar o cruzar con otras fuentes.
- **Bandas espectrales utilizadas**: NDVI = (B08 − B04)/(B08 + B04), NDWI = (B03 − B08)/(B03 + B08), y NDCI = (B05 − B04)/(B05 + B04) —el índice detrás de la estimación de clorofila-a—, tal como se definieron en `notebooks/analisis_lab04.ipynb` (sección 3).
- **Índice y categoría de cianobacteria**: se lee el raster de clorofila-a estimada ya calculado en la Parte I (`notebooks/resultados_cyano/<lago>_<fecha>.tif`, flujo CyanoLakes/NDCI) y se clasifica con el *WHO Alert Levels Framework* para floraciones de cianobacterias en agua recreativa: `< 10 µg/L` bajo (vigilancia), `10–50 µg/L` moderado (alerta 1), `≥ 50 µg/L` alto (alerta 2).
- **Metadatos de la escena**: nubosidad (%) y satélite Sentinel-2 de origen, tomados de la tabla de fechas oficiales de `notebooks/analisis_lab04.ipynb` (sección 2).

Primero se verifica que los 22 rásteres de clorofila-a comparten exactamente la misma rejilla (CRS, transform, dimensiones) que sus NDVI/NDWI, para poder indexarlos con la misma fila/columna sin reproyectar ni remuestrear.

In [ ]:
for (lago, fecha), grupo in inventario.groupby(["lago", "fecha"]):
    ruta_ndvi = grupo.loc[grupo["indice"] == "ndvi", "ruta"].iloc[0]
    ruta_chl = CYANO_DIR / f"{lago}_{fecha.strftime('%Y-%m-%d')}.tif"
    assert ruta_chl.exists(), f"Falta el raster de clorofila-a: {ruta_chl}"
    with rasterio.open(ruta_ndvi) as src_ndvi, rasterio.open(ruta_chl) as src_chl:
        assert src_ndvi.crs == src_chl.crs, f"CRS distinto en {ruta_chl}"
        assert src_ndvi.transform == src_chl.transform, f"Transform distinto en {ruta_chl}"
        assert src_ndvi.shape == src_chl.shape, f"Dimensiones distintas en {ruta_chl}"

print("Los 22 rásteres de clorofila-a comparten rejilla con sus NDVI/NDWI correspondientes.")

BANDAS_INDICES = {
    "ndvi": "B08,B04",
    "ndwi": "B03,B08",
    "chl_cyano": "B05,B04",
}

METADATOS_ESCENAS = pd.DataFrame([
    ("atitlan", "2025-01-18", 0.02, "Sentinel-2B"),
    ("atitlan", "2025-04-13", 0.54, "Sentinel-2C"),
    ("atitlan", "2025-05-13", 4.37, "Sentinel-2C"),
    ("atitlan", "2025-07-17", 3.57, "Sentinel-2A"),
    ("atitlan", "2025-11-21", 3.15, "Sentinel-2A"),
    ("atitlan", "2025-12-29", 3.17, "Sentinel-2C"),
    ("atitlan", "2026-02-12", 0.04, "Sentinel-2B"),
    ("atitlan", "2026-03-24", 3.17, "Sentinel-2B"),
    ("atitlan", "2026-04-13", 0.01, "Sentinel-2B"),
    ("atitlan", "2026-04-28", 4.96, "Sentinel-2C"),
    ("atitlan", "2026-07-22", 4.02, "Sentinel-2B"),
    ("amatitlan", "2025-01-28", 0.06, "Sentinel-2B"),
    ("amatitlan", "2025-04-15", 0.09, "Sentinel-2A"),
    ("amatitlan", "2025-04-28", 1.03, "Sentinel-2B"),
    ("amatitlan", "2025-11-24", 0.50, "Sentinel-2B"),
    ("amatitlan", "2026-01-08", 0.77, "Sentinel-2C"),
    ("amatitlan", "2026-02-02", 0.39, "Sentinel-2B"),
    ("amatitlan", "2026-02-07", 0.02, "Sentinel-2C"),
    ("amatitlan", "2026-03-29", 0.01, "Sentinel-2C"),
    ("amatitlan", "2026-04-13", 0.09, "Sentinel-2B"),
    ("amatitlan", "2026-04-28", 4.96, "Sentinel-2C"),
    ("amatitlan", "2026-06-19", 13.00, "Sentinel-2A"),
], columns=["lago", "fecha", "nubosidad_pct", "satelite"]).set_index(["lago", "fecha"])

escenas_inventario = set(
    inventario.assign(fecha_str=inventario["fecha"].dt.strftime("%Y-%m-%d"))[["lago", "fecha_str"]]
    .drop_duplicates()
    .itertuples(index=False, name=None)
)
assert set(METADATOS_ESCENAS.index) == escenas_inventario, \
    "La tabla de metadatos no cubre exactamente las 22 escenas del inventario"

METADATOS_ESCENAS.head()

### Criterios de limpieza

Los GeoTIFF de NDVI/NDWI cubren toda la caja delimitadora de cada lago (incluye tierra) y, como se vio en la sección 6, no filtran denominadores cercanos a cero. Antes de aplanarlos a una tabla de píxeles se descartan, en este orden:

1. **NoData**: píxeles `NaN` en NDVI o NDWI (bordes de la escena sin datos).
2. **Fuera de rango físico**: NDVI o NDWI fuera de `[-1, 1]` (visto en la sección 6; típicamente bordes de nube o de la caja delimitadora).
3. **Fuera del lago (tierra/vegetación)**: no existe un polígono de límites de lago ni una máscara de agua en `data/external/`, así que se construye una máscara espectral simple — `NDWI > 0` y `NDVI < 0.1` — siguiendo el criterio clásico de McFeeters para cuerpos de agua, la misma base que usa la máscara CyanoLakes de la Parte I.
4. **Nubes/sombras residuales dentro del agua**: dentro de la máscara de agua de cada escena, valores de NDVI o NDWI a más de 4 desviaciones estándar de la media de esa escena se excluyen como posibles artefactos. El umbral es deliberadamente amplio (4σ) para no recortar floraciones reales, que sí pueden desplazar el NDVI/NDWI de una escena.

In [ ]:
def construir_tabla_pixeles(
    lago: str,
    fecha: pd.Timestamp,
    ruta_ndvi: Path,
    ruta_ndwi: Path,
    ruta_chl: Path,
) -> pd.DataFrame:
    """Aplana un trío NDVI/NDWI/clorofila-a a una tabla de píxeles limpia y enriquecida."""
    with rasterio.open(ruta_ndvi) as src:
        ndvi = src.read(1)
        transform = src.transform
        crs = src.crs
    with rasterio.open(ruta_ndwi) as src:
        ndwi = src.read(1)
    with rasterio.open(ruta_chl) as src:
        chl = src.read(1).astype("float32")

    # Mismo criterio que resultados_temporales_cianobacteria.csv (Parte I): NoData -> NaN,
    # negativos (no interpretables físicamente) -> 0.
    chl[~np.isfinite(chl)] = np.nan
    chl[chl < 0] = 0

    filas, columnas = np.indices(ndvi.shape)
    xs, ys = rasterio.transform.xy(transform, filas.ravel(), columnas.ravel())
    fecha_str = fecha.strftime("%Y-%m-%d")
    metadatos_escena = METADATOS_ESCENAS.loc[(lago, fecha_str)]

    tabla = pd.DataFrame({
        "lago": lago,
        "fecha": fecha,
        "fila": filas.ravel().astype("int32"),
        "columna": columnas.ravel().astype("int32"),
        "x": np.asarray(xs, dtype="float32"),
        "y": np.asarray(ys, dtype="float32"),
        "ndvi": ndvi.ravel().astype("float32"),
        "bandas_ndvi": BANDAS_INDICES["ndvi"],
        "ndwi": ndwi.ravel().astype("float32"),
        "bandas_ndwi": BANDAS_INDICES["ndwi"],
        "chl_cyano": chl.ravel(),
        "bandas_chl_cyano": BANDAS_INDICES["chl_cyano"],
        "nubosidad_pct": metadatos_escena["nubosidad_pct"],
        "satelite": metadatos_escena["satelite"],
    })

    # 1) NoData
    sin_nodata = tabla["ndvi"].notna() & tabla["ndwi"].notna()

    # 2) Rango físico válido de índices normalizados
    en_rango = tabla["ndvi"].between(-1, 1) & tabla["ndwi"].between(-1, 1)

    # 3) Máscara espectral de agua (McFeeters): excluye tierra y vegetación
    es_agua = (tabla["ndwi"] > 0) & (tabla["ndvi"] < 0.1)

    tabla_limpia = tabla.loc[sin_nodata & en_rango & es_agua].copy()

    # 4) Outliers residuales (posibles nubes/sombras) dentro del agua de esta escena
    for columna_indice in ("ndvi", "ndwi"):
        media = tabla_limpia[columna_indice].mean()
        desviacion = tabla_limpia[columna_indice].std()
        limite = 4 * desviacion
        tabla_limpia = tabla_limpia[(tabla_limpia[columna_indice] - media).abs() <= limite]

    # Coordenadas geográficas: se transforman solo las filas ya limpias (más rápido)
    lons, lats = rasterio.warp.transform(
        crs, "EPSG:4326", tabla_limpia["x"].to_numpy(), tabla_limpia["y"].to_numpy()
    )
    tabla_limpia["lon"] = np.asarray(lons, dtype="float32")
    tabla_limpia["lat"] = np.asarray(lats, dtype="float32")

    # Categoría de cianobacteria: WHO Alert Levels Framework sobre clorofila-a (µg/L)
    tabla_limpia["categoria_cyano"] = pd.cut(
        tabla_limpia["chl_cyano"],
        bins=[-np.inf, 10, 50, np.inf],
        labels=["bajo (vigilancia)", "moderado (alerta 1)", "alto (alerta 2)"],
    )

    columnas_finales = [
        "lago", "fecha", "fila", "columna", "x", "y", "lon", "lat",
        "ndvi", "bandas_ndvi", "ndwi", "bandas_ndwi",
        "chl_cyano", "bandas_chl_cyano", "categoria_cyano",
        "nubosidad_pct", "satelite",
    ]
    return tabla_limpia[columnas_finales]

###  Construcción del dataset completo

Se aplica `construir_tabla_pixeles` a cada combinación lago/fecha (22 escenas) y se concatenan los resultados. Se registra, por escena, cuántos píxeles se conservaron respecto al total original.

In [ ]:
tablas = []
resumen_limpieza = []

for (lago, fecha), grupo in inventario.groupby(["lago", "fecha"]):
    ruta_ndvi = grupo.loc[grupo["indice"] == "ndvi", "ruta"].iloc[0]
    ruta_ndwi = grupo.loc[grupo["indice"] == "ndwi", "ruta"].iloc[0]
    ruta_chl = CYANO_DIR / f"{lago}_{fecha.strftime('%Y-%m-%d')}.tif"
    pixeles_totales = int(grupo["pixeles_totales"].iloc[0])

    tabla_limpia = construir_tabla_pixeles(lago, fecha, ruta_ndvi, ruta_ndwi, ruta_chl)
    tablas.append(tabla_limpia)

    resumen_limpieza.append({
        "lago": lago,
        "fecha": fecha.strftime("%Y-%m-%d"),
        "pixeles_totales": pixeles_totales,
        "pixeles_limpios": len(tabla_limpia),
        "pct_conservado": 100 * len(tabla_limpia) / pixeles_totales,
        "pct_con_categoria_cyano": (
            100 * tabla_limpia["categoria_cyano"].notna().mean() if len(tabla_limpia) else 0.0
        ),
    })

dataset_pixeles = pd.concat(tablas, ignore_index=True)
dataset_pixeles["lago"] = dataset_pixeles["lago"].astype("category")
dataset_pixeles["satelite"] = dataset_pixeles["satelite"].astype("category")
resumen_limpieza = pd.DataFrame(resumen_limpieza)

pixeles_totales_global = int(inventario.drop_duplicates(["lago", "fecha"])["pixeles_totales"].sum())
print(f"Escenas procesadas: {len(resumen_limpieza)}")
print(f"Píxeles originales (22 escenas): {pixeles_totales_global:,}")
print(f"Filas en el dataset limpio: {len(dataset_pixeles):,} ({100 * len(dataset_pixeles) / pixeles_totales_global:.2f} % del total)")
print(
    "Filas con categoría de cianobacteria asignada: "
    f"{dataset_pixeles['categoria_cyano'].notna().sum():,} "
    f"({100 * dataset_pixeles['categoria_cyano'].notna().mean():.2f} %)"
)
resumen_limpieza.sort_values("pct_conservado")

###  Verificación visual de la limpieza

Se compara el NDWI original contra el resultado tras aplicar los cuatro criterios de limpieza, para la misma escena de referencia usada en la sección 7.

In [ ]:
lago_muestra, fecha_muestra_ts = "atitlan", pd.Timestamp("2026-04-28")

fila_ndvi = inventario[
    (inventario["lago"] == lago_muestra)
    & (inventario["fecha"] == fecha_muestra_ts)
    & (inventario["indice"] == "ndvi")
].iloc[0]
fila_ndwi = inventario[
    (inventario["lago"] == lago_muestra)
    & (inventario["fecha"] == fecha_muestra_ts)
    & (inventario["indice"] == "ndwi")
].iloc[0]

with rasterio.open(fila_ndvi["ruta"]) as src:
    ndvi_muestra = src.read(1)
with rasterio.open(fila_ndwi["ruta"]) as src:
    ndwi_muestra = src.read(1)

ruta_chl_muestra = CYANO_DIR / f"{lago_muestra}_{fecha_muestra_ts.strftime('%Y-%m-%d')}.tif"
with rasterio.open(ruta_chl_muestra) as src:
    chl_muestra = src.read(1).astype("float32")
chl_muestra[~np.isfinite(chl_muestra)] = np.nan
chl_muestra[chl_muestra < 0] = 0

mascara_valida = (
    ~np.isnan(ndvi_muestra) & ~np.isnan(ndwi_muestra)
    & (ndvi_muestra >= -1) & (ndvi_muestra <= 1)
    & (ndwi_muestra >= -1) & (ndwi_muestra <= 1)
    & (ndwi_muestra > 0) & (ndvi_muestra < 0.1)
)

categoria_muestra = np.select(
    [chl_muestra < 10, (chl_muestra >= 10) & (chl_muestra < 50), chl_muestra >= 50],
    [1.0, 2.0, 3.0],
    default=np.nan,
)
categoria_muestra[~mascara_valida] = np.nan

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(ndwi_muestra, cmap="Blues", vmin=-1, vmax=1)
axes[0].set_title(f"NDWI original — {lago_muestra} {fecha_muestra_ts.date()}")
axes[0].axis("off")

ndwi_filtrado = np.where(mascara_valida, ndwi_muestra, np.nan)
axes[1].imshow(ndwi_filtrado, cmap="Blues", vmin=-1, vmax=1)
axes[1].set_title("NDWI tras limpieza (solo agua válida)")
axes[1].axis("off")

cmap_categorias = ListedColormap(["#2ca02c", "#ff7f0e", "#d62728"])
imagen_categorias = axes[2].imshow(categoria_muestra, cmap=cmap_categorias, vmin=1, vmax=3)
axes[2].set_title("Categoría de cianobacteria (WHO)")
axes[2].axis("off")
colorbar = fig.colorbar(imagen_categorias, ax=axes[2], ticks=[1, 2, 3], shrink=0.8)
colorbar.ax.set_yticklabels(["Bajo", "Moderado", "Alto"])

plt.show()

print(f"Píxeles conservados (agua válida): {mascara_valida.sum():,} de {ndwi_muestra.size:,} ({100 * mascara_valida.sum() / ndwi_muestra.size:.2f} %)")
print(
    "De esos, con estimación de clorofila-a disponible: "
    f"{np.isfinite(categoria_muestra).sum():,} "
    f"({100 * np.isfinite(categoria_muestra).sum() / mascara_valida.sum():.2f} %)"
)

### Guardar el dataset limpio

El resultado (una fila por píxel de agua válido, con su posición e índices espectrales) se guarda en Parquet — más compacto y rápido de leer que CSV para los ~millones de filas del dataset — junto con el resumen de limpieza por escena.

In [ ]:
ruta_dataset = OUTPUT_DIR / "dataset_pixeles_limpio.parquet"
dataset_pixeles.to_parquet(ruta_dataset, index=False)

ruta_resumen_limpieza = OUTPUT_DIR / "resumen_limpieza_pixeles.csv"
resumen_limpieza.to_csv(ruta_resumen_limpieza, index=False)

print(f"Dataset de píxeles limpio: {ruta_dataset.resolve()} ({len(dataset_pixeles):,} filas, {ruta_dataset.stat().st_size / 1e6:.1f} MB)")
print(f"Resumen de limpieza por escena: {ruta_resumen_limpieza.resolve()}")
print(f"Columnas: {list(dataset_pixeles.columns)}")
dataset_pixeles.head()

### Subconjunto listo para entrenamiento supervisado

`categoria_cyano` (y `chl_cyano`) quedan en `NaN` en las filas donde la máscara de agua NDVI/NDWI (usada aquí) marca el píxel como agua, pero la máscara CyanoLakes (usada en la Parte I para generar el raster de clorofila-a) no — son dos aproximaciones espectrales distintas del mismo lago y no coinciden al 100 %.

Eso importa en el momento de entrenar: si `categoria_cyano` (o `chl_cyano`) es la variable objetivo de un modelo de ML/DL, ningún framework puede calcular una pérdida contra una etiqueta `NaN`. La práctica correcta es **descartar esas filas**, no imputarlas — rellenar una categoría de cianobacteria que nunca se midió inventaría una etiqueta falsa. Si en cambio se usan NDVI/NDWI como *features* para otro fin (no para predecir cianobacteria), el dataset completo (`dataset_pixeles_limpio.parquet`) sigue siendo válido tal cual.

Por eso se genera aquí un segundo archivo, ya filtrado a solo las filas con etiqueta válida, junto con el balance de clases — importante porque si una categoría domina el dataset (lo esperable: la mayoría de los píxeles de agua no están en floración), el entrenamiento necesitará class weights, muestreo estratificado u otra técnica para no sesgarse hacia la clase mayoritaria.

In [ ]:
dataset_supervisado = dataset_pixeles.dropna(subset=["categoria_cyano"]).copy()

print(f"Filas totales (dataset_pixeles_limpio): {len(dataset_pixeles):,}")
print(
    f"Filas con categoría de cianobacteria (dataset_supervisado): {len(dataset_supervisado):,} "
    f"({100 * len(dataset_supervisado) / len(dataset_pixeles):.2f} %)"
)

balance_categorias = (
    dataset_supervisado["categoria_cyano"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .rename("pct_del_subconjunto")
)
print("\nBalance de clases:")
print(balance_categorias)

ruta_dataset_supervisado = OUTPUT_DIR / "dataset_pixeles_supervisado.parquet"
dataset_supervisado.to_parquet(ruta_dataset_supervisado, index=False)
print(f"\nDataset listo para entrenamiento supervisado: {ruta_dataset_supervisado.resolve()}")

###  Resumen del dataset

In [ ]:
print(f"Observaciones totales: {len(dataset_pixeles):,}")
print(f"Lagos: {dataset_pixeles['lago'].nunique()} ")
print(f"Escenas (lago x fecha): {dataset_pixeles.groupby(['lago', 'fecha'], observed=True).ngroups}")
print(f"Fechas de calendario unicas: {dataset_pixeles['fecha'].nunique()} (menos de 22 porque 2 fechas coinciden entre Atitlan y Amatitlan)")

In [ ]:
dataset_pixeles.groupby("lago", observed=True).size().rename("observaciones").to_frame()

In [ ]:
dataset_pixeles.groupby(["lago", "fecha"], observed=True).size().rename("observaciones").reset_index()

In [ ]:
resumen_variables = pd.DataFrame({
    "tipo_dato": dataset_pixeles.dtypes.astype(str),
    "pct_faltantes": (dataset_pixeles.isna().mean() * 100).round(3),
})
resumen_variables

### Análisis exploratorio de variables

`ndvi` y `ndwi` son las variables predictoras; `chl_cyano`/`categoria_cyano` es la variable objetivo. Las visualizaciones usan una muestra aleatoria de 200,000 filas (las estadísticas y correlaciones se calculan sobre el dataset completo).

In [ ]:
dataset_pixeles[["ndvi", "ndwi", "chl_cyano"]].describe()

In [ ]:
muestra_eda = dataset_pixeles.sample(n=200_000, random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, columna in zip(axes, ["ndvi", "ndwi", "chl_cyano"]):
    ax.hist(muestra_eda[columna].dropna(), bins=60, color="#277DA1")
    ax.set_title(columna.upper())
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
muestra_eda.boxplot(column="ndvi", by="categoria_cyano", ax=axes[0])
muestra_eda.boxplot(column="ndwi", by="categoria_cyano", ax=axes[1])
fig.suptitle("")
axes[0].set_title("NDVI por categoría de cianobacteria")
axes[1].set_title("NDWI por categoría de cianobacteria")
plt.tight_layout()
plt.show()

In [ ]:
correlaciones = dataset_pixeles[["ndvi", "ndwi", "chl_cyano", "nubosidad_pct"]].corr()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(correlaciones, vmin=-1, vmax=1, cmap="RdBu_r")
ax.set_xticks(range(len(correlaciones.columns)))
ax.set_xticklabels(correlaciones.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlaciones.columns)))
ax.set_yticklabels(correlaciones.columns)
for i in range(len(correlaciones)):
    for j in range(len(correlaciones)):
        ax.text(j, i, f"{correlaciones.iloc[i, j]:.2f}", ha="center", va="center")
fig.colorbar(im, shrink=0.8)
plt.tight_layout()
plt.show()

### NDVI casi no explica la cianobacteria por sí solo, y ni NDWI la explica fuertemente

In [ ]:
balance_por_lago = (
    dataset_supervisado.groupby(["lago", "categoria_cyano"], observed=True)
    .size()
    .groupby(level=0, group_keys=False, observed=True)
    .apply(lambda serie: 100 * serie / serie.sum())
    .unstack()
)

balance_por_lago.plot(kind="bar", figsize=(8, 5), color=["#2ca02c", "#ff7f0e", "#d62728"])
plt.ylabel("% de observaciones")
plt.title("Balance de clases por lago")
plt.tight_layout()
plt.show()

### Confirma visualmente el desbalance severo que ya habíamos visto (~99.5% "bajo") y si ese desbalance es parejo entre los dos lagos o si uno tiene más floraciones que el otro.

In [ ]:
promedio_por_fecha = (
    dataset_pixeles.groupby(["lago", "fecha"], observed=True)[["ndvi", "ndwi", "chl_cyano"]]
    .mean()
    .reset_index()
)

fig, ax = plt.subplots(figsize=(12, 5))
for lago, grupo in promedio_por_fecha.groupby("lago", observed=True):
    ax.plot(grupo["fecha"], grupo["chl_cyano"], marker="o", label=lago)
ax.set_ylabel("Clorofila-a promedio (µg/L)")
ax.set_title("Clorofila-a promedio por fecha (agregada desde el dataset de píxeles)")
ax.legend(title="Lago")
plt.tight_layout()
plt.show()

### Podemos ver ocmo la clorofila en lso diferentes lagos ha aumentado a lo largo del tiempo

### Decisiones de preparación y limpieza

- **Unidad de observación**: cada fila es un píxel de 10×10 m (no un raster completo), para poder entrenar modelos a nivel de píxel; por eso el dataset tiene millones de filas a partir de solo 44 GeoTIFF.
- **Descarte de `NoData`**: píxeles `NaN` en NDVI o NDWI no aportan información y se eliminan.
- **Rango físico `[-1, 1]`**: `src/descargar_indices.py` no filtra denominadores cercanos a cero, así que algunos píxeles de borde quedan fuera del rango matemáticamente válido de un índice normalizado; se eliminan por no ser interpretables.
- **Máscara de agua por umbral espectral (`NDWI > 0` y `NDVI < 0.1`)**: no existe un polígono oficial del límite del lago en `data/external/`, así que se aproxima con el criterio clásico de McFeeters para distinguir agua de tierra/vegetación.
- **Outliers a 4σ dentro de la máscara de agua**: para descartar posibles nubes/sombras residuales sin recortar floraciones reales, que también pueden desplazar el NDVI/NDWI de una escena; se usa un umbral amplio a propósito.
- **`chl_cyano` negativo → 0**: mismo criterio que ya usaba `resultados_temporales_cianobacteria.csv` en la Parte I, para no descartar el píxel completo por un artefacto de signo.
- **Categorización con el *WHO Alert Levels Framework***: umbrales de clorofila-a (`<10`, `10–50`, `≥50` µg/L) estandarizados para floraciones de cianobacterias, en vez de un umbral arbitrario.
- **`categoria_cyano` en `NaN` se descarta, no se imputa**, para el subconjunto de entrenamiento supervisado (`dataset_pixeles_supervisado.parquet`): inventar una etiqueta de cianobacteria falsearía el modelo. El dataset completo (`dataset_pixeles_limpio.parquet`) se conserva para otros usos.
- **Formato Parquet en vez de CSV**: con más de 12 millones de filas, Parquet es mucho más compacto y conserva los tipos de dato (categorías, fechas) sin tener que volver a declararlos al leerlo.
- **Los `.parquet`/`.csv` derivados no se versionan en git** (`**/data/processed/` en `.gitignore`): son reproducibles corriendo este notebook, igual que el resto de los datos procesados del proyecto.

---

# Ejercicio 2 — Construcción de la variable respuesta

En esta sección se transforma el índice continuo de cianobacteria preparado en el ejercicio 1 en una respuesta binaria. Todos los cálculos parten de `dataset_pixeles` y se restringen a observaciones con `chl_cyano` válido; una etiqueta ausente no se imputa porque hacerlo inventaría la variable objetivo.


## 2.1 y 2.2 Variable binaria y justificación del punto de corte

Se define `alta_cyano = 0` cuando `chl_cyano < 12 µg/L` y `alta_cyano = 1` cuando `chl_cyano >= 12 µg/L`. La clase 1 debe interpretarse como **presencia elevada que alcanza un nivel de alerta**, no como confirmación de toxicidad.

La elección de **12 µg/L** sigue el marco actualizado de la Organización Mundial de la Salud para aguas recreativas. La OMS (2021) ubica la vigilancia entre 1 y 12 µg/L de clorofila-a cuando dominan las cianobacterias, y el nivel de alerta 1 entre 12 y 24 µg/L; el nivel de alerta 2 se activa por evidencia de espuma superficial (*scum*) o transparencia muy baja, condiciones que este raster por sí solo no mide. Por eso, 12 µg/L es el límite numérico científicamente respaldado que separa vigilancia de una condición que requiere respuesta de gestión.

Los cortes 10 y 50 µg/L utilizados en la categorización exploratoria del ejercicio 1 corresponden al marco histórico de la OMS de 2003. La guía de 2021 declara que reemplaza aquella edición y actualiza los niveles de biomasa. Para esta respuesta final se usa el marco vigente, sin modificar la variable exploratoria anterior.

**Limitación ambiental:** `chl_cyano` es una estimación satelital de clorofila-a basada en NDCI y se usa como indicador de biomasa. No mide directamente recuentos celulares, biovolumen ni cianotoxinas; por lo tanto, una predicción positiva señala una zona prioritaria para verificación y muestreo de campo.

**Bibliografía**

- World Health Organization. (2021). *Guidelines on recreational water quality: Volume 1 — Coastal and fresh waters*. ISBN 978-92-4-003130-2. https://www.who.int/publications/i/item/9789240031302
- World Health Organization. (2021). *Guidelines on Recreational Water Quality*, resumen ejecutivo: niveles de vigilancia (1–12 µg/L) y alerta 1 (12–24 µg/L) para clorofila-a con dominancia de cianobacterias. https://www.ncbi.nlm.nih.gov/books/NBK572625/
- World Health Organization. (2003). *Guidelines for safe recreational water environments, Volume 1*. Referencia histórica de los umbrales 10/50 µg/L, reemplazada por la guía de 2021. https://www.who.int/publications/i/item/9241545801


In [ ]:
UMBRAL_ALERTA_CHL = 12.0  # µg/L, OMS (2021), inicio del nivel de alerta 1
NOMBRE_RESPUESTA = "alta_cyano"

dataset_respuesta = dataset_pixeles.dropna(subset=["chl_cyano"]).copy()
dataset_respuesta[NOMBRE_RESPUESTA] = (
    dataset_respuesta["chl_cyano"] >= UMBRAL_ALERTA_CHL
).astype("uint8")

assert dataset_respuesta[NOMBRE_RESPUESTA].notna().all()
assert set(dataset_respuesta[NOMBRE_RESPUESTA].unique()).issubset({0, 1})
assert len(dataset_respuesta) == dataset_pixeles["chl_cyano"].notna().sum()

print(f"Umbral: {UMBRAL_ALERTA_CHL:g} µg/L")
print("0 = ausencia/baja presencia (< 12 µg/L)")
print("1 = presencia elevada / nivel de alerta (>= 12 µg/L)")
print(f"Observaciones etiquetadas: {len(dataset_respuesta):,}")
dataset_respuesta[["chl_cyano", NOMBRE_RESPUESTA]].head()


## 2.3 Distribución de la respuesta

Se reportan conteos y porcentajes globales, por lago y por cada escena lago–fecha. Mostrar ambos es importante: los conteos reflejan el número de píxeles, mientras que los porcentajes permiten comparar lagos y fechas con distinta superficie válida.


In [ ]:
conteo_global = (
    dataset_respuesta[NOMBRE_RESPUESTA]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis(NOMBRE_RESPUESTA)
    .rename("observaciones")
)
resumen_global = conteo_global.to_frame()
resumen_global["porcentaje"] = 100 * resumen_global["observaciones"] / resumen_global["observaciones"].sum()
resumen_global.index = ["0: ausencia/baja", "1: elevada/alerta"]
display(resumen_global.style.format({"observaciones": "{:,}", "porcentaje": "{:.4f}%"}))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
colores = ["#4C78A8", "#E45756"]
resumen_global["observaciones"].plot(kind="bar", ax=axes[0], color=colores)
axes[0].set_title("Distribución global (conteo)")
axes[0].set_ylabel("Píxeles")
axes[0].tick_params(axis="x", rotation=0)
resumen_global["porcentaje"].plot(kind="bar", ax=axes[1], color=colores)
axes[1].set_title("Distribución global (porcentaje)")
axes[1].set_ylabel("%")
axes[1].tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
conteo_lago = (
    dataset_respuesta.groupby(["lago", NOMBRE_RESPUESTA], observed=True)
    .size().unstack(fill_value=0).reindex(columns=[0, 1], fill_value=0)
)
conteo_lago.columns = ["n_clase_0", "n_clase_1"]
conteo_lago["total"] = conteo_lago.sum(axis=1)
conteo_lago["pct_clase_0"] = 100 * conteo_lago["n_clase_0"] / conteo_lago["total"]
conteo_lago["pct_clase_1"] = 100 * conteo_lago["n_clase_1"] / conteo_lago["total"]
display(conteo_lago.style.format({
    "n_clase_0": "{:,}", "n_clase_1": "{:,}", "total": "{:,}",
    "pct_clase_0": "{:.4f}%", "pct_clase_1": "{:.4f}%",
}))

conteo_lago[["pct_clase_0", "pct_clase_1"]].plot(
    kind="bar", stacked=True, figsize=(8, 5), color=["#4C78A8", "#E45756"]
)
plt.ylabel("% de píxeles etiquetados")
plt.xlabel("Lago")
plt.title("Distribución de la respuesta por lago")
plt.legend(["0: ausencia/baja", "1: elevada/alerta"], loc="upper right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
conteo_fecha = (
    dataset_respuesta.groupby(["lago", "fecha", NOMBRE_RESPUESTA], observed=True)
    .size().unstack(fill_value=0).reindex(columns=[0, 1], fill_value=0)
)
conteo_fecha.columns = ["n_clase_0", "n_clase_1"]
conteo_fecha["total"] = conteo_fecha.sum(axis=1)
conteo_fecha["pct_clase_1"] = 100 * conteo_fecha["n_clase_1"] / conteo_fecha["total"]
resumen_fecha = conteo_fecha.reset_index().sort_values(["lago", "fecha"])
display(resumen_fecha.style.format({
    "n_clase_0": "{:,}", "n_clase_1": "{:,}", "total": "{:,}",
    "pct_clase_1": "{:.4f}%",
}))

fig, ax = plt.subplots(figsize=(12, 5))
for lago, grupo in resumen_fecha.groupby("lago", observed=True):
    ax.plot(grupo["fecha"], grupo["pct_clase_1"], marker="o", linewidth=2, label=lago.capitalize())
ax.set_ylabel("Píxeles en clase 1 (%)")
ax.set_xlabel("Fecha")
ax.set_title("Presencia elevada de cianobacteria por lago y fecha")
ax.legend(title="Lago")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 2.4 Desbalance entre clases

Además de observar los porcentajes, se cuantifica la razón entre la clase mayoritaria y la minoritaria. La exactitud de un clasificador ingenuo que siempre predice la clase mayoritaria sirve como línea base: un modelo útil debe superarla en métricas sensibles a la clase 1.


In [ ]:
n0, n1 = conteo_global.loc[0], conteo_global.loc[1]
n_mayoritaria, n_minoritaria = max(n0, n1), min(n0, n1)
razon_desbalance = np.inf if n_minoritaria == 0 else n_mayoritaria / n_minoritaria
exactitud_ingenua = 100 * n_mayoritaria / (n0 + n1)

print(f"Razón clase mayoritaria : minoritaria = {razon_desbalance:,.2f} : 1")
print(f"Exactitud del clasificador ingenuo (siempre mayoría) = {exactitud_ingenua:.4f}%")
if razon_desbalance >= 10:
    print("Conclusión: existe un desbalance severo (razón >= 10:1).")
elif razon_desbalance >= 3:
    print("Conclusión: existe un desbalance moderado (razón >= 3:1).")
else:
    print("Conclusión: no se observa un desbalance importante bajo esta regla descriptiva.")


Un desbalance grande puede hacer que el entrenamiento favorezca la clase 0: el modelo minimiza la pérdida acertando muchos píxeles comunes, aunque no detecte las zonas ambientales de interés. También vuelve engañosa la *accuracy*, porque predecir siempre la clase mayoritaria puede producir un valor alto sin recuperar ningún positivo.

En los ejercicios de modelado se deberá: (1) separar entrenamiento y prueba por grupos espaciales y/o por escena, no por píxeles vecinos al azar; (2) estratificar en la medida compatible con esos grupos; (3) considerar pesos de clase o remuestreo **solo dentro del conjunto de entrenamiento**; y (4) evaluar *recall* de la clase 1, precisión, F1, matriz de confusión, balanced accuracy y PR-AUC. ROC-AUC puede reportarse, pero PR-AUC suele ser más informativa cuando la clase positiva es muy rara. El umbral de decisión del modelo deberá elegirse con el costo ambiental de falsos negativos en mente.


## 2.5 Variables que producirían fuga de información

La respuesta `alta_cyano` es una comparación determinista de `chl_cyano` contra 12 µg/L. A su vez, `chl_cyano` proviene del índice NDCI calculado con B05 y B04. Por tanto, se excluyen `chl_cyano`, cualquier categoría construida a partir de él, NDCI y sus bandas fuente B04/B05 si llegaran a incorporarse como columnas numéricas. Permitir cualquiera de esas variables haría que el modelo reconstruyera la regla de etiquetado en lugar de aprender una relación generalizable.

`bandas_chl_cyano` también se excluye: es metadato constante que documenta la fórmula, no una característica espectral por píxel. `fila`/`columna` y un segundo sistema de coordenadas no son fuga de la respuesta, pero son redundantes; se conserva solamente `lon`/`lat`. NDVI y NDWI no son transformaciones de `chl_cyano` y se mantienen como candidatos, aunque esta decisión deberá declararse: NDVI comparte B04 con NDCI, por lo que una variante conservadora del experimento puede repetir el modelado sin NDVI como análisis de sensibilidad.


In [ ]:
variables_prohibidas = {
    NOMBRE_RESPUESTA,       # variable objetivo
    "chl_cyano",          # fuente directa de la etiqueta
    "categoria_cyano",    # otra discretización de la misma fuente
    "bandas_chl_cyano",   # metadato del cálculo, sin variación por píxel
    "ndci", "B04", "B05",  # índice fuente y bandas que lo construyen, si existieran
}

predictores_candidatos = [
    "lon", "lat", "fecha", "lago", "ndvi", "ndwi",
    "nubosidad_pct", "satelite",
]
predictores_candidatos = [c for c in predictores_candidatos if c in dataset_respuesta.columns]
assert variables_prohibidas.isdisjoint(predictores_candidatos)

auditoria_predictores = pd.DataFrame({
    "variable": sorted(set(dataset_respuesta.columns) | {"ndci", "B04", "B05"}),
})
auditoria_predictores["decision"] = auditoria_predictores["variable"].map(
    lambda c: "EXCLUIR: fuga/derivación del objetivo" if c in variables_prohibidas
    else ("CANDIDATO" if c in predictores_candidatos else "EXCLUIR: identificador, redundante o metadato")
)
display(auditoria_predictores)
print("Predictores candidatos:", predictores_candidatos)


In [ ]:
# Se conserva la fuente continua para trazabilidad, pero deberá eliminarse de X antes de entrenar.
ruta_respuesta = OUTPUT_DIR / "dataset_respuesta_binaria.parquet"
dataset_respuesta.to_parquet(ruta_respuesta, index=False)

ruta_balance_fecha = OUTPUT_DIR / "distribucion_respuesta_por_lago_fecha.csv"
resumen_fecha.to_csv(ruta_balance_fecha, index=False)

print(f"Dataset con respuesta binaria: {ruta_respuesta.resolve()}")
print(f"Resumen por lago y fecha: {ruta_balance_fecha.resolve()}")


## Conclusión del ejercicio 2

La variable respuesta queda definida de forma reproducible y respaldada por el marco OMS 2021. Los resúmenes anteriores permiten comprobar su frecuencia global, espacial y temporal, y cuantifican el desbalance sin depender de porcentajes redondeados. Para el modelado, la principal precaución será evitar simultáneamente la fuga espectral del índice que generó la etiqueta y la fuga espacial/temporal producida por separar al azar píxeles vecinos de una misma escena.
